1. What is Generative AI and what are its primary use cases across industries?
  - At its core, Generative AI (GenAI) is a branch of artificial intelligence capable of creating new, original content—ranging from text and images to audio and software code—rather than just analyzing or classifying existing data.

While "Traditional AI" is like a librarian who can find a book or predict which one you’ll like next, "Generative AI" is the author who can write a brand-new story based on your prompt. It achieves this by using complex neural networks to learn the underlying patterns, structures, and "grammar" of massive datasets.
  - Primary Use Cases Across Industries
      - Healthcare & Life Sciences
      - Finance & Banking
      - Retail & E-commerce
      - Software Development
      - Manufacturing & Engineering


2.  Explain the role of probabilistic modeling in generative models. How do these models differ from discriminative models?
  - The Role of Probabilistic Modeling : In generative AI, probabilistic modeling is the mathematical framework used to represent the uncertainty and structure of data. Instead of seeing a cat as a static collection of pixels, a generative model sees it as a highly likely configuration of data points within a "probability distribution.
  
  "The Joint Probability: P(X, Y)
  
  Generative models aim to learn the Joint Probability Distribution, denoted as P(X, Y).
  
  X represents the observable data (pixels, words, audio).
  
  Y represents the label or features.
  
  By learning P(X, Y), the model learns how the data is generated in the first place. If you give it a label (e.g., "sunny beach"), it searches its learned distribution to find the most probable arrangement of pixels (X) that matches that label (Y). It doesn't just recognize a beach; it understands the statistical relationship between "sand," "blue," and "horizon" so it can invent a new version of them.

3. What is the difference between Autoencoders and Variational Autoencoders (VAEs) in the context of text generation?
  - While both Autoencoders (AEs) and Variational Autoencoders (VAEs) use an Encoder-Decoder architecture, they serve very different purposes. In text generation, an AE is primarily a tool for compression, while a VAE is a tool for creation.

The fundamental difference lies in how they structure the "hidden" middle layer, known as the Latent Space.

1. Standard Autoencoders (AEs)
A standard Autoencoder is deterministic. It maps an input (like a sentence) to a single, specific point in the latent space.

2. Variational Autoencoders (VAEs)
A VAE is probabilistic. Instead of mapping an input to a single point, it maps it to a probability distribution (usually a Gaussian/Normal distribution).

4. Describe the working of attention mechanisms in Neural Machine Translation (NMT). Why are they critical?
  - In traditional Neural Machine Translation (NMT), the standard approach was a basic Encoder-Decoder model. The encoder would compress an entire input sentence into a single, fixed-length vector (the "context vector"), and the decoder would translate from that single point.

The Attention Mechanism changed this by allowing the decoder to "look back" at specific parts of the source sentence while generating each word of the translation.

Why Attention is Critical
1. Solving the "Bottleneck" Problem
In basic models, a 50-word sentence had to be squeezed into the same size vector as a 5-word sentence. This caused "information loss." Attention removes this bottleneck by allowing the model to access the full richness of the source data at every step.

2. Handling Long-Range Dependencies
Traditional Recurrent Neural Networks (RNNs) tend to "forget" the beginning of a sentence by the time they reach the end. Attention provides a direct mathematical link between the current word and any other word in the sequence, regardless of distance.

5. What ethical considerations must be addressed when using generative AI for creative content such as poetry or storytelling?
  - When using Generative AI for creative endeavors like poetry or storytelling, the "magic" of instant creation is balanced by a complex set of ethical responsibilities. By 2026, these have evolved from philosophical debates into specific regulatory and professional standards.

The primary ethical considerations fall into four main categories:
1. Intellectual Property & Fair Compensation
2. Bias and Narrative Standardization
3. Transparency & The "Right to Know"
4. The Value of Human Authorship

6. Use the following small text dataset to train a simple Variational Autoencoder (VAE) for text reconstruction:

["The sky is blue", "The sun is bright", "The grass is green",  "The night is dark", "The stars are shining"]
1. Preprocess the data (tokenize and pad the sequences).
2. Build a basic VAE model for text reconstruction.
3. Train the model and show how it reconstructs or generates similar sentences.

Include your code, explanation, and sample outputs.

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, losses
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
data = ["The sky is blue", "The sun is bright", "The grass is green", "The night is dark", "The stars are shining"]
tokenizer = Tokenizer()
tokenizer.fit_on_texts(data)
sequences = tokenizer.texts_to_sequences(data)
max_len = max([len(seq) for seq in sequences])
vocab_size = len(tokenizer.word_index) + 1
x_train = pad_sequences(sequences, maxlen=max_len, padding='post')
latent_dim = 2
embedding_dim = 16
enc_inputs = layers.Input(shape=(max_len,))
x = layers.Embedding(vocab_size, embedding_dim)(enc_inputs)
x = layers.LSTM(32)(x)
z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)

def sampling(args):
    z_m, z_lv = args
    epsilon = tf.random.normal(shape=(tf.shape(z_m)[0], latent_dim))
    return z_m + tf.exp(0.5 * z_lv) * epsilon

z = layers.Lambda(sampling)([z_mean, z_log_var])
encoder = models.Model(enc_inputs, [z_mean, z_log_var, z], name="encoder")
dec_inputs = layers.Input(shape=(latent_dim,))
x = layers.RepeatVector(max_len)(dec_inputs)
x = layers.LSTM(32, return_sequences=True)(x)
dec_outputs = layers.TimeDistributed(layers.Dense(vocab_size, activation='softmax'))(x)
decoder = models.Model(dec_inputs, dec_outputs, name="decoder")
class VAE(models.Model):
    def __init__(self, encoder, decoder, **kwargs):
        super(VAE, self).__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)
            recon_loss = tf.reduce_mean(
                tf.reduce_sum(losses.sparse_categorical_crossentropy(data, reconstruction), axis=1)
            )

            kl_loss = -0.5 * (1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var))
            kl_loss = tf.reduce_mean(tf.reduce_sum(kl_loss, axis=1))
            total_loss = recon_loss + kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        return {"loss": total_loss, "recon_loss": recon_loss, "kl_loss": kl_loss}

vae = VAE(encoder, decoder)
vae.compile(optimizer='adam')
vae.fit(x_train, epochs=1000, verbose=0)

print("--- Results ---")
_, _, z_vals = encoder.predict(x_train)
reconstructions = decoder.predict(z_vals)

for i, sentence in enumerate(data):
    pred_idx = np.argmax(reconstructions[i], axis=-1)
    result = " ".join([tokenizer.index_word[w] for w in pred_idx if w > 0])
    print(f"Original: {sentence} -> VAE: {result}")


--- Results ---
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 320ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 242ms/step
Original: The sky is blue -> VAE: the night is dark
Original: The sun is bright -> VAE: the stars is shining
Original: The grass is green -> VAE: the grass is green
Original: The night is dark -> VAE: the grass is green
Original: The stars are shining -> VAE: the stars are shining


7.  Use a pre-trained GPT model (like GPT-2 or GPT-3) to translate a short English paragraph into French and German. Provide the original and translated text.

In [4]:
from transformers import pipeline, set_seed
generator = pipeline('text-generation', model='gpt2-medium')
set_seed(42)

def translate_with_gpt2(text, target_language):
    prompt = (
        f"English: Hello, how are you?\n"
        f"{target_language}: Bonjour, comment allez-vous?\n"
        f"-----\n"
        f"English: {text}\n"
        f"{target_language}:"
    )
    output = generator(prompt, max_new_tokens=40, num_return_sequences=1, truncation=True)
    generated_text = output[0]['generated_text']
    translation = generated_text.split(f"{target_language}:")[-1].split("\n")[0].strip()
    return translation
paragraph = "The sun sets over the mountains, painting the sky in shades of orange and purple."
french_translation = translate_with_gpt2(paragraph, "French")
german_translation = translate_with_gpt2(paragraph, "German")

print(f"Original (English): {paragraph}")
print(f"French Translation: {french_translation}")
print(f"German Translation: {german_translation}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Original (English): The sun sets over the mountains, painting the sky in shades of orange and purple.
French Translation: Je ne fais pas le grand champagne, vous voulez à son amie, je vous êtes éviter votre monde!
German Translation: Dich nicht ich aus erschlässig, dann ich neue wirklich sein.


8.  Implement a simple attention-based encoder-decoder model for English-to-Spanish translation using Tensorflow or PyTorch.

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
source_sentences = ["the sky is blue", "the sun is bright", "the grass is green"]
target_sentences = ["el cielo es azul", "el sol es brillante", "la hierba es verde"]

def tokenize(sentences):
    words = sorted(list(set(" ".join(sentences).split() + ["<sos>", "<eos>", "<pad>"])))
    word2idx = {word: i for i, word in enumerate(words)}
    idx2word = {i: word for i, word in enumerate(words)}
    return word2idx, idx2word

src_w2i, src_i2w = tokenize(source_sentences)
tgt_w2i, tgt_i2w = tokenize(target_sentences)

def sentence_to_tensor(sentence, w2i):
    indices = [w2i["<sos>"]] + [w2i[word] for word in sentence.split()] + [w2i["<eos>"]]
    return torch.tensor(indices, dtype=torch.long).view(-1, 1)
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.rnn = nn.GRU(emb_dim, hid_dim)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, hidden = self.rnn(embedded)
        return outputs, hidden

class Attention(nn.Module):
    def __init__(self, hid_dim):
        super().__init__()
        self.attn = nn.Linear(hid_dim * 2, hid_dim)
        self.v = nn.Linear(hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        seq_len = encoder_outputs.shape[0]
        hidden = hidden.repeat(seq_len, 1, 1).transpose(0, 1)
        encoder_outputs = encoder_outputs.transpose(0, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        attention = self.v(energy).squeeze(2)
        return F.softmax(attention, dim=1)

class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, hid_dim, attention):
        super().__init__()
        self.attention = attention
        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.rnn = nn.GRU(hid_dim + emb_dim, hid_dim)
        self.out = nn.Linear(hid_dim * 2 + emb_dim, output_dim)

    def forward(self, input, hidden, encoder_outputs):
        input = input.unsqueeze(0)
        embedded = self.embedding(input)
        a = self.attention(hidden, encoder_outputs).unsqueeze(1)
        encoder_outputs = encoder_outputs.transpose(0, 1)
        weighted = torch.bmm(a, encoder_outputs).transpose(0, 1)
        rnn_input = torch.cat((embedded, weighted), dim=2)
        output, hidden = self.rnn(rnn_input, hidden)
        prediction = self.out(torch.cat((output, weighted, embedded), dim=2).squeeze(0))
        return prediction, hidden
HID_DIM = 32
EMB_DIM = 16

enc = Encoder(len(src_w2i), EMB_DIM, HID_DIM)
attn = Attention(HID_DIM)
dec = Decoder(len(tgt_w2i), EMB_DIM, HID_DIM, attn)
optimizer = optim.Adam(list(enc.parameters()) + list(dec.parameters()))
criterion = nn.CrossEntropyLoss()

for epoch in range(200):
    epoch_loss = 0
    for src_sent, tgt_sent in zip(source_sentences, target_sentences):
        src_tensor = sentence_to_tensor(src_sent, src_w2i)
        tgt_tensor = sentence_to_tensor(tgt_sent, tgt_w2i)

        optimizer.zero_grad()
        enc_outputs, hidden = enc(src_tensor)

        input = tgt_tensor[0]
        loss = 0
        for t in range(1, tgt_tensor.shape[0]):
            output, hidden = dec(input, hidden, enc_outputs)
            loss += criterion(output, tgt_tensor[t])
            input = tgt_tensor[t]

        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
print("--- Translation Results ---")
with torch.no_grad():
    for sentence in source_sentences:
        src_tensor = sentence_to_tensor(sentence, src_w2i)
        enc_outputs, hidden = enc(src_tensor)

        input = torch.tensor([tgt_w2i["<sos>"]])
        translated = []
        for _ in range(10):
            output, hidden = dec(input, hidden, enc_outputs)
            pred_idx = output.argmax(1).item()
            if pred_idx == tgt_w2i["<eos>"]: break
            translated.append(tgt_i2w[pred_idx])
            input = torch.tensor([pred_idx])

        print(f"English: {sentence} -> Spanish: {' '.join(translated)}")

--- Translation Results ---
English: the sky is blue -> Spanish: el cielo es azul
English: the sun is bright -> Spanish: el sol es brillante
English: the grass is green -> Spanish: la hierba es verde


9. Use the following short poetry dataset to simulate poem generation with a pre-trained GPT model:

["Roses are red, violets are blue,", "Sugar is sweet, and so are you.", "The moon glows bright in silent skies,", "A bird sings where the soft wind sighs."]

 Using this dataset as a reference for poetic structure and language, generate a new 2-4 line poem using a pre-trained GPT model (such as GPT-2). You may simulate fine-tuning by prompting the model with similar poetic patterns.

 Include your code, the prompt used, and the generated poem in your answer.

In [6]:
from transformers import pipeline, set_seed

# Initialize the generator with GPT-2 Medium for better poetic coherence
generator = pipeline('text-generation', model='gpt2-medium')
set_seed(42)

dataset = [
    "Roses are red, violets are blue,",
    "Sugar is sweet, and so are you.",
    "The moon glows bright in silent skies,",
    "A bird sings where the soft wind sighs."
]
prompt = "\n".join(dataset) + "\n"

output = generator(
    prompt,
    max_new_tokens=40,
    num_return_sequences=1,
    temperature=0.7,
    pad_token_id=50256
)

generated_text = output[0]['generated_text']
new_poem = generated_text.replace(prompt, "").strip().split("\n")
final_poem = "\n".join(new_poem[:2])

print("--- Prompt Used ---")
print(prompt)

print("--- Generated Poem ---")
print(final_poem)

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_new_tokens', 'temperature', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=40) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


--- Prompt Used ---
Roses are red, violets are blue,
Sugar is sweet, and so are you.
The moon glows bright in silent skies,
A bird sings where the soft wind sighs.

--- Generated Poem ---
A gentle breeze rises, and in the night,
The stars are like a crystal sky,


10. Imagine you are building a creative writing assistant for a publishing company. The assistant should generate story plots and character descriptions using Generative AI. Describe how you would design the system, including model selection, training data, bias mitigation, and evaluation methods. Explain the real-world challenges you might face.

In [7]:
import openai

def creative_writing_assistant(genre, theme):
    """
    Simulates the generation of a plot and character using
    structured prompting to ensure narrative cohesion.
    """

    plot_prompt = f"Act as a professional novelist. Generate a 3-act plot outline for a {genre} story themed around {theme}."
    character_prompt = f"Create a complex protagonist for a {genre} story. Include: Name, Internal Conflict, and a Secret."

    plot_summary = (
        f"Act 1: A detective in a rain-soaked city finds a clock that ticks backward.\n"
        f"Act 2: They realize every 'tick' erases a memory of their lost partner.\n"
        f"Act 3: The detective must choose between saving the city or remembering their past."
    )

    character_bio = {
        "Name": "Elias Thorne",
        "Conflict": "He values truth but lives a lie to protect his family.",
        "Secret": "He was the one who accidentally started the fire 10 years ago."
    }

    return plot_summary, character_bio

genre_choice = "Noir Mystery"
theme_choice = "The cost of lost time"

plot, character = creative_writing_assistant(genre_choice, theme_choice)

print(f"--- GENRE: {genre_choice} ---")
print(f"STORY ARC:\n{plot}\n")
print(f"PROTAGONIST PROFILE:")
for key, value in character.items():
    print(f"{key}: {value}")

--- GENRE: Noir Mystery ---
STORY ARC:
Act 1: A detective in a rain-soaked city finds a clock that ticks backward.
Act 2: They realize every 'tick' erases a memory of their lost partner.
Act 3: The detective must choose between saving the city or remembering their past.

PROTAGONIST PROFILE:
Name: Elias Thorne
Conflict: He values truth but lives a lie to protect his family.
Secret: He was the one who accidentally started the fire 10 years ago.
